# VFD Crystallisation — Platform Mapping Demo

Demonstrates platform-specific parameter fitting for:
1. Qubit-like system (n=2)
2. Multi-mode system (n=4, analog oscillator)
3. Control-response curve generation
4. Cross-platform parameter comparison

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from vfd.crystallisation.synthetic_data import (
    ModelParams, ExperimentConfig, generate_vfd_dataset,
    generate_misspecified_dataset, generate_control_response_curve,
    simulate_observables,
)
from vfd.crystallisation.estimation import (
    fit_parameters, sensitivity_analysis, compare_models,
)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Qubit-Like Platform (n=2)

In [ ]:
theta_qubit = ModelParams(alpha=1.0, beta=0.3, gamma=0.5, eta=0.01, Gamma=0.1)
config_qubit = ExperimentConfig(n_modes=2, n_trials=200, n_basins=2, max_steps=500, seed=42)

ds_qubit = generate_vfd_dataset(theta_qubit, config_qubit)
print('Qubit platform observables:')
for k, v in ds_qubit.summary.items():
    if k != 'basin_preference':
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'  basin_preference: {ds_qubit.summary["basin_preference"].round(3)}')

## 2. Analog Oscillator Platform (n=4)

In [ ]:
theta_analog = ModelParams(alpha=1.5, beta=0.5, gamma=1.0, eta=0.005, Gamma=0.0, kappa_B=1.2)
config_analog = ExperimentConfig(n_modes=4, n_trials=200, n_basins=4, max_steps=500, seed=99)

ds_analog = generate_vfd_dataset(theta_analog, config_analog)
print('Analog platform observables:')
for k, v in ds_analog.summary.items():
    if k != 'basin_preference':
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'  basin_preference: {ds_analog.summary["basin_preference"].round(3)}')

## 3. Control-Response Curves

Sweep Gamma (environment coupling) and observe outcome entropy.

In [ ]:
gamma_vals = np.linspace(0.0, 3.0, 8)
config_sweep = ExperimentConfig(n_modes=3, n_trials=100, n_basins=3, max_steps=500, seed=42)

xs, ys = generate_control_response_curve(
    theta_qubit, 'Gamma', gamma_vals, 'outcome_entropy', config_sweep
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, ys, 'o-', color='steelblue', markersize=8)
ax.set_xlabel('Gamma (environment coupling)')
ax.set_ylabel('Outcome entropy H')
ax.set_title('Control-Response: Entropy vs Environment Coupling')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('As Gamma increases, noise drives system toward stochastic selection (higher entropy).')

## 4. False Positive Guard: Fitting VFD to Non-VFD Data

In [ ]:
# Generate data from standard decoherence (no VFD dynamics)
config_fp = ExperimentConfig(n_modes=3, n_trials=200, n_basins=3, max_steps=500, seed=42)
ds_dec = generate_misspecified_dataset(config_fp, model='decoherence')

# Fit VFD model to it
fit_dec = fit_parameters(ds_dec, config_fp)
print(f'VFD fit on decoherence data: residual = {fit_dec.residual_norm:.4f}')

# Fit VFD to VFD data for comparison
ds_vfd = generate_vfd_dataset(theta_qubit, config_fp)
fit_vfd = fit_parameters(ds_vfd, config_fp, theta_init=theta_qubit)
print(f'VFD fit on VFD data: residual = {fit_vfd.residual_norm:.4f}')

print(f'\nIf VFD fit on decoherence data has similar residual, VFD terms are not justified.')